
# Efficient Local Attention + Span-Hypergraph + HCA-Style Compressed Memory LM

This notebook trains a small decoder-only language model on a larger streaming LLM dataset.

Architecture:

```text
Token embedding
→ local causal attention block(s)
→ O(T) causal span-hypergraph blocks
→ optional HCA-style compressed global-memory attention blocks
→ LayerNorm
→ tied LM head
```

The goal is to avoid dense full-context attention. The default dataset is **FineWeb-Edu `sample-10BT`**, streamed from Hugging Face.

Main complexity:

- local attention: approximately `O(T * local_window)` conceptually, though this simple notebook uses SDPA with a local mask for convenience.
- span-hypergraph blocks: `O(T * num_widths * d)`.
- compressed memory attention: `O(T * (T / compression_block))`, much cheaper than full `O(T^2)` when the compression block is large.

For true 100k+ sequence experiments, replace the local attention implementation with a block-sparse/sliding-window FlashAttention kernel. The rest of the model is structured to be long-context-friendly.


## 1. Install dependencies

Run this cell once if your environment does not already have `datasets`, `transformers`, and `accelerate` installed.

In [1]:

# Uncomment if needed:
# %pip install -q datasets transformers accelerate


## 2. Imports and configuration

In [2]:

import math
import os
import time
from dataclasses import dataclass
from typing import Iterable, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

try:
    from datasets import load_dataset
    from transformers import AutoTokenizer
except ImportError as e:
    raise ImportError("Install dependencies with: pip install datasets transformers accelerate") from e

# -----------------------
# User-editable settings
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 1337

# Dataset: FineWeb-Edu sample-10BT is much larger than Tiny Shakespeare.
# You can swap to e.g. dataset_name="roneneldan/TinyStories" for a smaller/debug dataset.
dataset_name = "HuggingFaceFW/fineweb-edu"
dataset_config = "sample-10BT"
dataset_split = "train"
text_field = "text"
tokenizer_name = "gpt2"
streaming = True
shuffle_buffer = 50_000
val_docs = 2_000  # held out by taking the first docs from the stream

# Training shape. Start modestly. Increase block_size once the notebook works.
batch_size = 4
block_size = 2048
num_workers = 0  # keep 0 in notebooks; increase in scripts if desired

# Model size.
n_embd = 384
n_head = 6
n_local_attn_layers = 1
n_span_layers = 6
n_compressed_memory_layers = 1  # inserted near the top of the stack
span_widths = (2, 4, 8, 16, 32, 64)
local_window = 256
compression_block = 64

# Optimization.
max_iters = 10_000
eval_interval = 500
eval_iters = 50
learning_rate = 3e-4
min_lr_ratio = 0.1
warmup_iters = 200
weight_decay = 0.1
grad_clip = 1.0
dropout = 0.1
use_amp = True and device.startswith("cuda")
use_compile = False
save_best_checkpoint = True
checkpoint_path = "best_hca_span_hypergraph_lm.pt"

# Generation.
generate_tokens = 400
temperature = 0.8
top_k = 50
prompt = "The meaning of intelligence is"

torch.manual_seed(seed)
if device.startswith("cuda"):
    torch.cuda.manual_seed_all(seed)
    print("GPU:", torch.cuda.get_device_name(0))
print("device:", device)

GPU: Tesla T4
device: cuda


## 3. Streaming packed-token dataset

This uses Hugging Face streaming, so the full dataset is not downloaded before training. Each iterator tokenizes documents and packs them into contiguous `block_size + 1` token chunks for next-token prediction.

In [3]:

class StreamingPackedTokenDataset(IterableDataset):
    def __init__(
        self,
        dataset_name: str,
        dataset_config: Optional[str],
        split: str,
        tokenizer_name: str,
        text_field: str,
        block_size: int,
        streaming: bool = True,
        shuffle: bool = True,
        shuffle_buffer: int = 10_000,
        seed: int = 0,
        skip_docs: int = 0,
        take_docs: Optional[int] = None,
    ):
        super().__init__()
        self.dataset_name = dataset_name
        self.dataset_config = dataset_config
        self.split = split
        self.tokenizer_name = tokenizer_name
        self.text_field = text_field
        self.block_size = block_size
        self.streaming = streaming
        self.shuffle = shuffle
        self.shuffle_buffer = shuffle_buffer
        self.seed = seed
        self.skip_docs = skip_docs
        self.take_docs = take_docs
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
        if self.tokenizer.eos_token_id is None:
            self.tokenizer.add_special_tokens({"eos_token": "<|endoftext|>"})
        self.eos_id = self.tokenizer.eos_token_id

    def _make_stream(self):
        if self.dataset_config is None:
            ds = load_dataset(self.dataset_name, split=self.split, streaming=self.streaming)
        else:
            ds = load_dataset(self.dataset_name, self.dataset_config, split=self.split, streaming=self.streaming)

        # Validation stream should be deterministic; training stream should be shuffled.
        if self.shuffle:
            ds = ds.shuffle(buffer_size=self.shuffle_buffer, seed=self.seed)

        if self.skip_docs:
            ds = ds.skip(self.skip_docs)
        if self.take_docs is not None:
            ds = ds.take(self.take_docs)
        return ds

    def __iter__(self):
        # Infinite stream for training; finite stream for validation when take_docs is set.
        while True:
            token_buffer = []
            yielded_any = False
            for row in self._make_stream():
                text = row.get(self.text_field, None)
                if not isinstance(text, str) or len(text) == 0:
                    continue
                ids = self.tokenizer.encode(text, add_special_tokens=False)
                ids.append(self.eos_id)
                token_buffer.extend(ids)

                while len(token_buffer) >= self.block_size + 1:
                    chunk = token_buffer[: self.block_size + 1]
                    token_buffer = token_buffer[self.block_size + 1 :]
                    x = torch.tensor(chunk[:-1], dtype=torch.long)
                    y = torch.tensor(chunk[1:], dtype=torch.long)
                    yielded_any = True
                    yield x, y

            if self.take_docs is not None:
                # Finite validation stream: stop after one pass.
                break
            if not yielded_any:
                raise RuntimeError("Dataset iterator yielded no examples. Check dataset config/text field.")


def make_loaders():
    # Hold out the first val_docs documents for validation. Training skips them.
    train_ds = StreamingPackedTokenDataset(
        dataset_name=dataset_name,
        dataset_config=dataset_config,
        split=dataset_split,
        tokenizer_name=tokenizer_name,
        text_field=text_field,
        block_size=block_size,
        streaming=streaming,
        shuffle=True,
        shuffle_buffer=shuffle_buffer,
        seed=seed,
        skip_docs=val_docs,
        take_docs=None,
    )
    val_ds = StreamingPackedTokenDataset(
        dataset_name=dataset_name,
        dataset_config=dataset_config,
        split=dataset_split,
        tokenizer_name=tokenizer_name,
        text_field=text_field,
        block_size=block_size,
        streaming=streaming,
        shuffle=False,
        seed=seed,
        skip_docs=0,
        take_docs=val_docs,
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=num_workers, pin_memory=device.startswith("cuda"))
    val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers, pin_memory=device.startswith("cuda"))
    return train_loader, val_loader, train_ds.tokenizer

train_loader, val_loader, tokenizer = make_loaders()
vocab_size = len(tokenizer)
print("vocab_size:", vocab_size)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)


vocab_size: 50257
eos token: <|endoftext|> 50256


## 4. Rotary position embeddings

In [4]:

def precompute_rope_cache(head_dim: int, max_seq_len: int, device, base: float = 10_000.0):
    assert head_dim % 2 == 0, "RoPE requires even head_dim"
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq_len, device=device).float()
    freqs = torch.einsum("t,d->td", t, inv_freq)
    cos = freqs.cos()[None, None, :, :]  # [1, 1, T, D/2]
    sin = freqs.sin()[None, None, :, :]
    return cos, sin


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    # x: [B, H, T, D]
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    cos = cos[:, :, : x.size(-2), :]
    sin = sin[:, :, : x.size(-2), :]
    y_even = x_even * cos - x_odd * sin
    y_odd = x_even * sin + x_odd * cos
    return torch.stack((y_even, y_odd), dim=-1).flatten(-2)


def apply_rope_at_positions(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, positions: torch.Tensor):
    # x: [B, H, M, D], positions: [M]
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    c = cos[:, :, positions, :]
    s = sin[:, :, positions, :]
    y_even = x_even * c - x_odd * s
    y_odd = x_even * s + x_odd * c
    return torch.stack((y_even, y_odd), dim=-1).flatten(-2)


## 5. Model blocks

All blocks use pre-LayerNorm and residual/skip connections. The span-hypergraph and compressed-memory branches use learnable residual gates initialized near zero.

In [5]:

@dataclass
class EfficientHGConfig:
    vocab_size: int
    block_size: int
    n_embd: int = 384
    n_head: int = 6
    n_local_attn_layers: int = 1
    n_span_layers: int = 6
    n_compressed_memory_layers: int = 1
    span_widths: tuple = (2, 4, 8, 16, 32, 64)
    local_window: int = 256
    compression_block: int = 64
    dropout: float = 0.1


class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class CausalLocalSelfAttention(nn.Module):
    def __init__(self, cfg: EfficientHGConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.local_window = cfg.local_window
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.out = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.dropout = cfg.dropout
        self.resid_drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        H, D = self.n_head, self.head_dim
        qkv = self.qkv(x).view(B, T, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # [B, H, T, D]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # Boolean SDPA mask: True means allowed. Causal + local window.
        pos = torch.arange(T, device=x.device)
        i = pos[:, None]
        j = pos[None, :]
        mask = (j <= i) & ((i - j) < self.local_window)
        mask = mask.view(1, 1, T, T)

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out(y))


class LocalAttentionBlock(nn.Module):
    def __init__(self, cfg: EfficientHGConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalLocalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg.n_embd, cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.ln1(x), cos, sin)
        x = x + self.mlp(self.ln2(x))
        return x


class CausalSpanHypergraphBlock(nn.Module):
    """Linear-time causal span-hypergraph block.

    Each position t receives compressed hyperedge states for causal spans ending at t:
      [t-w+1, ..., t] for w in span_widths.
    Prefix sums make each width O(T), and the number of widths is constant.
    """
    def __init__(self, cfg: EfficientHGConfig, gate_init: float = -3.0):
        super().__init__()
        self.widths = tuple(cfg.span_widths)
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.in_proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.edge_proj = nn.Linear(cfg.n_embd * len(self.widths), cfg.n_embd)
        self.out_proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.gate = nn.Parameter(torch.tensor(float(gate_init)))
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg.n_embd, cfg.dropout)

    @staticmethod
    def span_mean_from_prefix(h, prefix, width: int):
        B, T, C = h.shape
        ends = torch.arange(1, T + 1, device=h.device)
        starts = (ends - width).clamp_min(0)
        span_sum = prefix[:, ends, :] - prefix[:, starts, :]
        span_len = (ends - starts).to(h.dtype).view(1, T, 1)
        return span_sum / span_len

    def forward(self, x, cos=None, sin=None):
        residual = x
        h = self.in_proj(self.ln1(x))
        prefix = torch.cat([torch.zeros_like(h[:, :1, :]), h.cumsum(dim=1)], dim=1)
        spans = [self.span_mean_from_prefix(h, prefix, w) for w in self.widths]
        z = torch.cat(spans, dim=-1)
        z = self.edge_proj(z)
        z = F.gelu(z)
        z = self.drop(self.out_proj(z))
        x = residual + torch.sigmoid(self.gate) * z
        x = x + self.mlp(self.ln2(x))
        return x


class CausalCompressedMemoryAttentionBlock(nn.Module):
    """HCA-style compressed global memory block.

    Tokens are compressed into causal blocks of size `compression_block` using learned gated pooling.
    Each token attends to completed compressed blocks plus one learned null memory.

    This is reduced-quadratic: O(T * T/compression_block), not strictly O(T).
    """
    def __init__(self, cfg: EfficientHGConfig, gate_init: float = -4.0):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.compression_block = cfg.compression_block
        self.dropout = cfg.dropout

        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.q_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.k_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.v_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.pool_score = nn.Linear(cfg.n_embd, 1, bias=False)
        self.out = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
        self.gate = nn.Parameter(torch.tensor(float(gate_init)))

        self.null_k = nn.Parameter(torch.zeros(1, cfg.n_head, 1, self.head_dim))
        self.null_v = nn.Parameter(torch.zeros(1, cfg.n_head, 1, self.head_dim))
        nn.init.normal_(self.null_k, std=0.02)
        nn.init.normal_(self.null_v, std=0.02)

        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg.n_embd, cfg.dropout)

    def _compress_blocks(self, y):
        B, T, C = y.shape
        cb = self.compression_block
        pad_len = (-T) % cb
        if pad_len:
            y_pad = F.pad(y, (0, 0, 0, pad_len))
        else:
            y_pad = y
        Tp = y_pad.size(1)
        nb = Tp // cb
        chunks = y_pad.view(B, nb, cb, C)

        valid = torch.arange(Tp, device=y.device).view(nb, cb) < T
        score = self.pool_score(chunks).squeeze(-1)  # [B, nb, cb]
        score = score.masked_fill(~valid.view(1, nb, cb), torch.finfo(score.dtype).min)
        weight = F.softmax(score, dim=-1)
        mem = (weight.unsqueeze(-1) * chunks).sum(dim=2)  # [B, nb, C]
        return mem, nb

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        H, D = self.n_head, self.head_dim
        residual = x
        y = self.ln1(x)

        q = self.q_proj(y).view(B, T, H, D).transpose(1, 2)  # [B,H,T,D]
        q = apply_rope(q, cos, sin)

        mem, nb = self._compress_blocks(y)
        k = self.k_proj(mem).view(B, nb, H, D).transpose(1, 2)
        v = self.v_proj(mem).view(B, nb, H, D).transpose(1, 2)

        block_ends = (torch.arange(nb, device=x.device) + 1) * self.compression_block - 1
        block_ends = block_ends.clamp_max(T - 1)
        k = apply_rope_at_positions(k, cos, sin, block_ends)

        null_k = self.null_k.expand(B, -1, -1, -1)
        null_v = self.null_v.expand(B, -1, -1, -1)
        k = torch.cat([null_k, k], dim=2)
        v = torch.cat([null_v, v], dim=2)

        token_pos = torch.arange(T, device=x.device)[:, None]
        allow_blocks = block_ends[None, :] <= token_pos  # completed blocks only
        allow = torch.cat([torch.ones(T, 1, device=x.device, dtype=torch.bool), allow_blocks], dim=1)
        allow = allow.view(1, 1, T, nb + 1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=allow,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.drop(self.out(out))
        x = residual + torch.sigmoid(self.gate) * out
        x = x + self.mlp(self.ln2(x))
        return x

## 6. Full language model

In [6]:

class EfficientHypergraphLM(nn.Module):
    def __init__(self, cfg: EfficientHGConfig):
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)

        blocks = []
        for _ in range(cfg.n_local_attn_layers):
            blocks.append(LocalAttentionBlock(cfg))

        # Insert compressed-memory block(s) after a few span blocks.
        mem_insert_every = max(1, cfg.n_span_layers // max(1, cfg.n_compressed_memory_layers))
        mem_inserted = 0
        for i in range(cfg.n_span_layers):
            blocks.append(CausalSpanHypergraphBlock(cfg))
            if mem_inserted < cfg.n_compressed_memory_layers and ((i + 1) % mem_insert_every == 0):
                blocks.append(CausalCompressedMemoryAttentionBlock(cfg))
                mem_inserted += 1

        self.blocks = nn.ModuleList(blocks)
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size
        cos, sin = precompute_rope_cache(self.cfg.n_embd // self.cfg.n_head, T, idx.device)
        x = self.drop(self.token_embedding(idx))
        for block in self.blocks:
            x = block(x, cos, sin)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

cfg = EfficientHGConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_local_attn_layers=n_local_attn_layers,
    n_span_layers=n_span_layers,
    n_compressed_memory_layers=n_compressed_memory_layers,
    span_widths=span_widths,
    local_window=local_window,
    compression_block=compression_block,
    dropout=dropout,
)

model = EfficientHypergraphLM(cfg).to(device)
print(model)
print(f"parameters: {count_parameters(model)/1e6:.2f}M")

if use_compile:
    print("Compiling model...")
    model = torch.compile(model)

EfficientHypergraphLM(
  (token_embedding): Embedding(50257, 384)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0): LocalAttentionBlock(
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): CausalLocalSelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=False)
        (out): Linear(in_features=384, out_features=384, bias=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (1-6): 6 x CausalSpanHypergraphBlock(
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (in_proj): Linear(in_features=384, out_features=384, bias=Tr

## 7. Forward-pass smoke test

In [7]:

xb, yb = next(iter(train_loader))
xb = xb.to(device, non_blocking=True)
yb = yb.to(device, non_blocking=True)

with torch.no_grad():
    with torch.amp.autocast("cuda", enabled=use_amp):
        logits, loss = model(xb, yb)
print("x:", xb.shape, "logits:", logits.shape, "loss:", float(loss))

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1631 > 1024). Running this sequence through the model will result in indexing errors


x: torch.Size([4, 2048]) logits: torch.Size([4, 2048, 50257]) loss: 10.885735511779785


## 8. Training utilities

In [8]:

def get_lr(step):
    if step < warmup_iters:
        return learning_rate * step / max(1, warmup_iters)
    if step > max_iters:
        return learning_rate * min_lr_ratio
    decay_ratio = (step - warmup_iters) / max(1, max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * min_lr_ratio + coeff * (learning_rate - learning_rate * min_lr_ratio)


def cycle(loader):
    while True:
        for batch in loader:
            yield batch

train_iter = cycle(train_loader)

@torch.no_grad()
def estimate_loss(model, val_loader, eval_iters):
    model.eval()
    losses = []
    it = iter(val_loader)
    for _ in range(eval_iters):
        try:
            xb, yb = next(it)
        except StopIteration:
            it = iter(val_loader)
            xb, yb = next(it)
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            _, loss = model(xb, yb)
        losses.append(loss.detach())
    model.train()
    return torch.stack(losses).mean().item()


def print_gates(model):
    m = model._orig_mod if hasattr(model, "_orig_mod") else model
    for name, module in m.named_modules():
        if hasattr(module, "gate") and isinstance(module.gate, nn.Parameter):
            raw = float(module.gate.detach().cpu())
            sig = float(torch.sigmoid(module.gate.detach()).cpu())
            print(f"{name:45s} raw={raw:+.4f} sigmoid={sig:.4f}")


## 9. Training loop

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

device_type = "cuda" if str(device).startswith("cuda") else "cpu"
amp_enabled = bool(use_amp and device_type == "cuda")

scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

best_val = float("inf")
loss_ema = None
tokens_since_eval = 0
total_tokens = 0
t0 = time.time()

model.train()

for step in range(max_iters + 1):
    if step % eval_interval == 0 or step == max_iters:
        elapsed = time.time() - t0

        if step == 0:
            toks_per_sec = 0.0
        else:
            toks_per_sec = tokens_since_eval / max(elapsed, 1e-9)

        val_loss = estimate_loss(model, val_loader, eval_iters)
        train_loss_str = "nan" if loss_ema is None else f"{loss_ema:.4f}"
        print(
            f"step {step:6d} | "
            f"train_ema {train_loss_str} | "
            f"val {val_loss:.4f} | "
            f"lr {get_lr(step):.2e} | "
            f"tok/s {toks_per_sec:,.0f} | "
            f"tokens {total_tokens:,} | "
            f"elapsed {elapsed:.1f}s"
        )
        print_gates(model)

        t0 = time.time()
        tokens_since_eval = 0
        if save_best_checkpoint and val_loss < best_val:
            best_val = val_loss
            state_model = model._orig_mod if hasattr(model, "_orig_mod") else model
            torch.save(
                {
                    "model": state_model.state_dict(),
                    "config": cfg.__dict__,
                    "step": step,
                    "val_loss": val_loss,
                    "train_loss_ema": loss_ema,
                    "total_tokens": total_tokens,
                    "tokenizer_name": tokenizer_name,
                },
                checkpoint_path,
            )
            print(f"saved best checkpoint to {checkpoint_path} with val={best_val:.4f}")

    if step == max_iters:
        break

    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    xb, yb = next(train_iter)
    xb = xb.to(device, non_blocking=True)
    yb = yb.to(device, non_blocking=True)
    tokens_this_step = xb.numel()
    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast(device_type=device_type, enabled=amp_enabled):
        _, loss = model(xb, yb)

    if not torch.isfinite(loss):
        print(f"non-finite loss at step {step}: {loss.item()}")
        continue

    loss_value = loss.detach().float().item()
    if loss_ema is None:
        loss_ema = loss_value
    else:
        loss_ema = 0.99 * loss_ema + 0.01 * loss_value

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    tokens_since_eval += tokens_this_step
    total_tokens += tokens_this_step

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1055 > 1024). Running this sequence through the model will result in indexing errors


step      0 | train_ema nan | val 10.8744 | lr 0.00e+00 | tok/s 0 | tokens 0 | elapsed 0.0s
blocks.1                                      raw=-3.0000 sigmoid=0.0474
blocks.2                                      raw=-3.0000 sigmoid=0.0474
blocks.3                                      raw=-3.0000 sigmoid=0.0474
blocks.4                                      raw=-3.0000 sigmoid=0.0474
blocks.5                                      raw=-3.0000 sigmoid=0.0474
blocks.6                                      raw=-3.0000 sigmoid=0.0474
blocks.7                                      raw=-4.0000 sigmoid=0.0180
saved best checkpoint to best_hca_span_hypergraph_lm.pt with val=10.8744


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

step    500 | train_ema 6.6848 | val 6.4594 | lr 2.99e-04 | tok/s 23,669 | tokens 4,096,000 | elapsed 173.1s
blocks.1                                      raw=-2.9084 sigmoid=0.0517
blocks.2                                      raw=-2.9100 sigmoid=0.0517
blocks.3                                      raw=-2.9304 sigmoid=0.0507
blocks.4                                      raw=-2.9505 sigmoid=0.0497
blocks.5                                      raw=-2.9557 sigmoid=0.0495
blocks.6                                      raw=-2.9568 sigmoid=0.0494
blocks.7                                      raw=-3.9367 sigmoid=0.0191
saved best checkpoint to best_hca_span_hypergraph_lm.pt with val=6.4594


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

step   1000 | train_ema 6.0847 | val 6.0296 | lr 2.96e-04 | tok/s 24,287 | tokens 8,192,000 | elapsed 168.6s
blocks.1                                      raw=-2.8218 sigmoid=0.0562
blocks.2                                      raw=-2.8209 sigmoid=0.0562
blocks.3                                      raw=-2.8626 sigmoid=0.0540
blocks.4                                      raw=-2.8941 sigmoid=0.0524
blocks.5                                      raw=-2.9072 sigmoid=0.0518
blocks.6                                      raw=-2.9125 sigmoid=0.0515
blocks.7                                      raw=-3.8799 sigmoid=0.0202
saved best checkpoint to best_hca_span_hypergraph_lm.pt with val=6.0296


## 10. Generate text

In [ ]:

# Optionally load best checkpoint before generation.
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    state_model = model._orig_mod if hasattr(model, "_orig_mod") else model
    state_model.load_state_dict(ckpt["model"])
    print("loaded checkpoint", checkpoint_path, "val_loss", ckpt.get("val_loss"), "step", ckpt.get("step"))

ids = tokenizer.encode(prompt, add_special_tokens=False)
context = torch.tensor([ids], dtype=torch.long, device=device)
out = model.generate(context, max_new_tokens=generate_tokens, temperature=0.7, top_k=top_k)[0].tolist()
print(tokenizer.decode(out))

## 11. Practical scaling notes

For 100k-token contexts, do **not** use dense global attention. Use:

- local/sliding attention with a real block-sparse or FlashAttention sliding-window kernel,
- span-hypergraph layers for linear-time composition,
- compressed-memory blocks with large `compression_block`, e.g. 256–2048.

This notebook keeps the implementation readable. For production-scale training, the first optimization target is replacing the local mask SDPA with a true sliding-window attention kernel.